# Magic Hour | AI Voice Editor - production notebook

Upload a voice recording, pick an edit, listen to the result, download a WAV. One backend (v0.6.0) behind three features:

- **Audio enhancing (cleanup):** enhance / denoise / dereverb
- **Transcript editing:** replace / insert / delete words in what was said
- **Prompt editing:** emotion, voice style, whisper, speed, or a custom plain-English instruction

Run the steps in order: **1** install the backend, **2** mount Drive, **3** runtime + models, **4** start the worker, **5** run one edit, **6** listen and download. Steps 1-4 only set things up. Nothing generates until step 5, and step 5 only runs when you tick `RUN_EDIT` yourself.

One thing you will notice: step 1 contains a single very long line starting `backend_bytes = "..."`. That is the backend code itself, embedded in the notebook so it always installs the exact tested version (v0.6.0, sha-checked) without needing repo or network access to our code. It is supposed to be there. The cell is collapsed, so you never have to scroll through it.

## Transcript editor status (2026-11-20)

The correctable-transcript editor (transcribe once, correct, align, review, preview, save) lives in the repo as `auk_reliability/transcriptflow.py` + `forcedalign.py`. CPU suite: 534/534 green, including the fail-closed gates. No notebook step runs this lifecycle: step 5's `transcript_editor` mode is the classic replace / insert / delete span edit.

**Not yet verified:** one real representative edit through actual GPU generation for this new flow. The 2026-11-20 L4 run verified forced alignment on real speech (14/14 words, mean uncalibrated fit 0.9054) and exposed two lifecycle bugs now fixed in the repo (accept-ASR-as-is spans could never reach generation; measured-timing transplants always failed closed). Both fixes have regression tests but no GPU pass yet. Do not present the editor as finished.

**The step 1 embedded backend archive was rebuilt 2026-11-20 as v0.6.0 from repo main @ da085b1 and carries those fixes.** Re-run step 1 (and step 2 after it) before any new session.

Harness lessons from the 2026-11-20 run: write a stage status file at every stage boundary; unbuffered heartbeat prints; per-cell timeouts; avoid long keystroke-typed code cells; idle GPU billing only ends with Runtime > Disconnect and delete runtime, verified via Manage sessions = no active sessions.

In [ ]:
# @title Step 1. Install tools and the backend v0.6.0 (voice enhancer + transcript editor; no inference)
# v0.6.0 (archive rebuilt 2026-11-20 from repo main @ da085b1): one embedded, sha-pinned snapshot of the
# current backend. Adds over v0.5.1: the correctable-transcript editor lifecycle fixes (accept-ASR-as-is
# spans now carry worker-registry evidence and still pass segment ASR re-verification; measured-timing
# transplants compute the base-cut fit instead of always failing closed), the pinned wav2vec2 forced
# alignment module (forcedalign.py), and the 0.5s segment floor with explicit wider-context options.
# 534/534 CPU tests on 2026-11-20 (Linux). Real GPU generation for this new transcript-editor flow is
# NOT yet verified - see the status cell at the top. Qwen2.5-Omni-3B conditioner remains non-commercial.
# Repo: github.com/advayc/auk-production-backend.
import sys, subprocess, pathlib, base64, hashlib, zipfile, io, os
if not (3,11) <= sys.version_info[:2] <= (3,13):
    raise RuntimeError("Use a Python 3.11-3.13 Colab runtime.")
WORK_DIR = "/content/auk-production"  # @param {type:"string"}
COMMERCIAL_USE = False  # @param {type:"boolean"}
COMMERCIAL_LICENSE_CONFIRMED = False  # @param {type:"boolean"}
if COMMERCIAL_USE and not COMMERCIAL_LICENSE_CONFIRMED: raise RuntimeError("Obtain commercial model-license clearance first.")
ROOT=pathlib.Path(WORK_DIR).resolve()
if str(ROOT) in ("/", "/content", "/tmp", str(pathlib.Path.home())): raise ValueError("Use a dedicated working directory")
ROOT.mkdir(parents=True,exist_ok=True);os.umask(0o077)
subprocess.run(["apt-get","-qq","update"],check=True)
subprocess.run(["apt-get","-qq","install","-y","ca-certificates","curl","ffmpeg","libgomp1","libssl-dev","git","cmake","build-essential","python3-venv"],check=True)
BACKEND_SHA256="8f968f6b536a545d98efdb2b3fa29a0e768899b87007fd3e97f65fc88ac4af49"
backend_bytes=base64.b64decode("...")
BACKEND_ARCHIVE=ROOT/"auk_backend_v060_2026-11-20.zip";BACKEND_ARCHIVE.write_bytes(backend_bytes)
with zipfile.ZipFile(io.BytesIO(backend_bytes)) as z:
    for info in z.infolist():
        if not (ROOT/info.filename).resolve().is_relative_to(ROOT) or ((info.external_attr >> 16)&0o170000)==0o120000:raise RuntimeError("Unsafe backend ZIP")
    z.extractall(ROOT)
BACKEND_DIR=ROOT  # fixed 2026-10-18: embedded zip extracts flat (pyproject.toml at WORK_DIR root), no wrapper dir
import shutil; shutil.rmtree(BACKEND_DIR/"build", ignore_errors=True)  # never reuse stale build files
subprocess.run([sys.executable,"-m","pip","install","--no-deps",str(BACKEND_DIR)],check=True)
if "AUK_WORKER" in globals():AUK_WORKER.close();del AUK_WORKER
for name in list(sys.modules):
    if name=="auk_reliability" or name.startswith("auk_reliability."):del sys.modules[name]
import auk_reliability
assert auk_reliability.__version__=="0.6.0"
from auk_reliability import worker as _w, durable as _d
assert "whole_clip" in pathlib.Path(_w.__file__).read_text() and "only=None" in pathlib.Path(_d.__file__).read_text(), "Stale backend files were installed"
from auk_reliability import enhance as _e
assert "adaptive_full_wet" in pathlib.Path(_e.__file__).read_text(), "Stale backend files were installed"
from auk_reliability import transcriptflow as _tf
assert "asr_accepted" in pathlib.Path(_tf.__file__).read_text(), "Stale backend files were installed"
CHECKPOINT = None  # a checkpoint object from before this cell may point at the OLD backup; run P0 again (new backup folder)
print("Installed",auk_reliability.__version__,"exportable archive:",BACKEND_ARCHIVE)


In [ ]:
# @title Step 2. Mount Drive and set up the checkpoint (no generation)
ENABLE_DRIVE_BACKUP = True # @param {type:"boolean"}
RESTORE_CHECKPOINT = False # @param {type:"boolean"}
# Tick RESTORE_CHECKPOINT only on a recycled runtime. It reads RESTORE_FROM and never writes to it. BACKUP_DIR must be a different, new folder.
RESTORE_FROM = "/content/drive/MyDrive/auk-checkpoints/session-2026-10-16-v025-b" # @param {type:"string"}
BACKUP_DIR = "/content/drive/MyDrive/auk-checkpoints/session-2026-11-20-handoff" # @param {type:"string"}
DATASET_DIR = "/content/auk-public-v1" # @param {type:"string"}
RUNS_DIR = "/content/auk-runs" # @param {type:"string"}
CHECKPOINT = None
if ENABLE_DRIVE_BACKUP:
    from google.colab import drive
    from auk_reliability.durable import Checkpoint
    drive.mount('/content/drive')
    if RESTORE_CHECKPOINT:
        if __import__('os').path.realpath(RESTORE_FROM) == __import__('os').path.realpath(BACKUP_DIR): raise ValueError('BACKUP_DIR must be a different, new folder from RESTORE_FROM')
        # v0.2.5: restore only the corpus/runs from the old backup. The backend comes from P1, never from the old backup.
        old = Checkpoint(RESTORE_FROM, {"dataset":DATASET_DIR,"runs":RUNS_DIR})
        print(old.restore(only=("dataset","runs")))
        # v0.2.6: also bring back earlier request folders (hash-checked copy; skips identical files; stops if a local file differs).
        import json as _j, hashlib as _h
        _src = __import__("pathlib").Path(RESTORE_FROM)
        _recs = [r for r in _j.loads((_src/"checkpoint.json").read_text())["files"] if r["path"].startswith("backend/shared-requests/")]
        def _sha(p):
            h = _h.sha256()
            with open(p, "rb") as f:
                for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
            return h.hexdigest()
        _copied = 0
        for r in _recs:
            rel = r["path"].split("/", 1)[1]
            if ".." in rel.split("/"): raise ValueError("unsafe path")
            sp, tp = _src/r["path"], ROOT/rel
            if _sha(sp) != r["sha256"]: raise ValueError("Corrupt backup file " + r["path"])
            if tp.exists():
                if _sha(tp) != r["sha256"]: raise FileExistsError("Local file differs: " + str(tp))
                continue
            tp.parent.mkdir(parents=True, exist_ok=True); __import__("shutil").copyfile(sp, tp); _copied += 1
        print({"request_files_restored": _copied, "listed": len(_recs)})
        import glob, shutil, pathlib as _pl
        tools = sorted(glob.glob(RESTORE_FROM + "/backend/recovery-tools/*"))
        if tools:
            for f in _pl.Path(tools[-1]).glob("*.py"):
                if not (_pl.Path("/content")/f.name).exists(): shutil.copyfile(f, _pl.Path("/content")/f.name)
            print("Recovery tools copied from", tools[-1])
    CHECKPOINT = Checkpoint(BACKUP_DIR, {"backend":ROOT,"dataset":DATASET_DIR,"runs":RUNS_DIR})
    print(CHECKPOINT.sync())
else: print("Enable before generation. Pick a unique backup folder for a new experiment; reuse its exact path to restore. Virtualenvs/models are excluded and must be rebuilt/downloaded.")


In [ ]:
# @title Step 3. Set up the runtime and download the models (downloads, no generation)
from auk_reliability.core import Limits, gpu_details, setup_runtime, setup_models, command, atomic_json
import site, os, subprocess, sys
LIMITS=Limits();GPU=gpu_details();print(GPU)
if GPU["compute_capability"]!="7.5":raise RuntimeError("Select a T4 runtime for this prebuilt. Other GPUs require explicit source-build validation.")
if GPU["total_mib"]<14000:raise RuntimeError("At least roughly 16 GiB total VRAM required")
# audio.cpp SM75 release uses CUDA 12.8, including on Colab images that ship CUDA 13.
subprocess.run([sys.executable,"-m","pip","install","--quiet","nvidia-cublas-cu12==12.8.4.1","nvidia-cuda-runtime-cu12==12.8.90","nvidia-cufft-cu12==11.3.3.83"],check=True)
search=set(site.getsitepackages()+[site.getusersitepackages()]);libdirs=[]
for root in search:
    for component in ("cublas","cuda_runtime","cufft"):
        p=pathlib.Path(root)/"nvidia"/component/"lib"
        if p.is_dir():libdirs.append(str(p))
os.environ["LD_LIBRARY_PATH"]=":".join(libdirs+[os.environ.get("LD_LIBRARY_PATH","")])
MEMORY_SAVER=True
BINARY,RUNTIME_LOCK=setup_runtime(ROOT,mode="t4_prebuilt",gpu=GPU)
link_check=command(["ldd",str(BINARY)],timeout=30)
if "not found" in link_check:raise RuntimeError("Runtime has unresolved libraries: "+link_check)
MODEL_DIR,MODEL_LOCK=setup_models(ROOT,["base_f32","base_q8","flash_f16"])
PROVENANCE={"runtime":RUNTIME_LOCK,"model":MODEL_LOCK,"environment":{"python":sys.version,"gpu":GPU,"nvidia_smi":command(["nvidia-smi"]),"ffmpeg":command(["ffmpeg","-version"]).splitlines()[0],"backend_sha256":BACKEND_SHA256}}
atomic_json(ROOT/"environment.json",PROVENANCE)
print("Ready. No model generation started. First request includes cold startup.")


In [ ]:
# @title Step 4. Start the worker and the QA checks (downloads, no generation)
import pathlib
ROOT = pathlib.Path(ROOT) if isinstance(ROOT, str) else ROOT
ASR_LANGUAGE = "en"  # @param {type:"string"}
SETUP_SHARED_WORKER = True  # @param {type:"boolean"}
ENABLE_SPEAKER_QA = True  # @param {type:"boolean"}
GPU_HOURLY_RATE_USD = -1.0  # @param {type:"number"}
if SETUP_SHARED_WORKER:
    _need = [n for n in ("ROOT","LIMITS","GPU","MEMORY_SAVER","ASR_LANGUAGE","BINARY","MODEL_DIR") if n not in globals()]
    if _need:
        raise RuntimeError("Run steps 1 and 3 first (missing " + ", ".join(_need) + "). ")
    if "PROVENANCE" not in globals():
        PROVENANCE = {"runtime": RUNTIME_LOCK, "model": MODEL_LOCK, "environment": {"python":sys.version}}
    if "auk_reliability" not in sys.modules:
        raise RuntimeError("Run P1 first.")
    from auk_reliability.core import LocalAuKServer, setup_asr
    from auk_reliability.colab import setup_optional_speaker_qa
    from auk_reliability.qa import SemanticQAGate
    from auk_reliability.worker import ReliabilityWorker, configure_default_worker, edit_audio
    from auk_reliability.acceptance import configure_acceptance_suite, run_acceptance_suite, benchmark_profiles, recommend_profiles
    if "AUK_WORKER" in globals(): AUK_WORKER.close()
    shared_asr, shared_asr_fingerprint = setup_asr(pathlib.Path(ROOT) / "shared-qa", LIMITS, ASR_LANGUAGE)
    speaker_callback, speaker_fingerprint, speaker_setup = setup_optional_speaker_qa(
        pathlib.Path(ROOT) / "shared-speaker-qa", enabled=ENABLE_SPEAKER_QA)
    print("Speaker QA:", speaker_setup)
    if speaker_callback is None:
        raise RuntimeError("Speaker QA unavailable. Fix setup before reliability runs; unavailable identity QA never passes.")
    transport = LocalAuKServer(BINARY, MODEL_DIR, ROOT / "shared-runtime", LIMITS, MEMORY_SAVER)
    runtime_profile = {**PROVENANCE, "gpu": GPU, "simulation": False,
        "asr_fingerprint": shared_asr_fingerprint, "speaker_fingerprint": speaker_fingerprint, "speaker_setup": speaker_setup}
    if GPU_HOURLY_RATE_USD >= 0: runtime_profile["gpu_hourly_rate_usd"] = GPU_HOURLY_RATE_USD
    AUK_WORKER = ReliabilityWorker(transport, ROOT / "shared-requests",
        semantic_gate=SemanticQAGate(shared_asr, speaker_callback), runtime_profile=runtime_profile)
    configure_default_worker(AUK_WORKER)
    print("Shared worker configured. No generation started. Attribute QA still needs human review.")
else:
    print("Shared worker setup is off. Enable when ready for QA downloads/install.")


In [ ]:
# @title Step 5. Run one edit (tick RUN_EDIT yourself; nothing runs on Run all)
INPUT_AUDIO = ""  # @param {type:"string"}
RUN_EDIT = False  # @param {type:"boolean"}
MODE = "voice_enhancer"  # @param ["voice_enhancer", "transcript_editor", "voice_editor"]
# voice_enhancer (cleanup): enhance | denoise | dereverb
# transcript_editor: replace | insert | delete
# (classic span edit with checked segment text - the newer correctable-transcript
# lifecycle from the status cell above is backend-only for now, not a notebook step)
# voice_editor (prompt edits): emotion | voice_characteristics | whisper | speed | custom
OPERATION = "enhance"  # @param ["enhance", "denoise", "dereverb", "replace", "insert", "delete", "emotion", "voice_characteristics", "whisper", "speed", "custom"]
PROFILE = "base_f32"  # @param ["base_f32", "base_q8", "flash_f16"]
RIGHTS_CONFIRMED = False  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}
# Human-checked transcript of the recording. Every mode uses it for the automatic content check.
CHECKED_SOURCE_TEXT = ""  # @param {type:"string"}
# whole_clip: edit the whole <=15 s recording, no segment times needed.
# localized: your own checked segment times below.
TRANSCRIPT_SCOPE = "whole_clip"  # @param ["whole_clip", "localized"]
# Per-operation settings. Edit only the ones for your chosen OPERATION.
EMOTION = "calm"  # @param {type:"string"}
VOICE_DESCRIPTION = "warm, slightly deeper voice"  # @param {type:"string"}
SPEED_FACTOR = 1.25  # @param {type:"number"}
CUSTOM_INSTRUCTION = "Keep every word but sound more confident."  # @param {type:"string"}
# Transcript edits need a listened-to, nonempty interval (max 15 s) and the exact checked text inside it.
SEGMENT_START_SECONDS = 1.23  # @param {type:"number"}
SEGMENT_END_SECONDS = 2.05  # @param {type:"number"}
SEGMENT_TEXT = "the better team"  # @param {type:"string"}
# replace: phrase to replace / delete: phrase to remove / insert: text to add. Type it WITHOUT quote marks.
EDIT_TARGET_TEXT = ""  # @param {type:"string"}
# replace only. Type it WITHOUT quote marks.
EDIT_REPLACEMENT_TEXT = ""  # @param {type:"string"}
INSERT_ANCHOR = ""  # @param {type:"string"}
INSERT_POSITION = "before"  # @param ["before", "after"]

def _clean(v): return "" if v is None else str(v).strip().strip("\"'\u201c\u201d").strip()

def build_parameters():
    global EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT
    EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT = _clean(EDIT_TARGET_TEXT), _clean(EDIT_REPLACEMENT_TEXT)
    base = {"source_text": CHECKED_SOURCE_TEXT, "rights_confirmed": RIGHTS_CONFIRMED,
            "profile": PROFILE, "seed": SEED}
    op = OPERATION
    if MODE == "voice_enhancer":
        if op not in ("enhance", "denoise", "dereverb"):
            raise RuntimeError(f"{op} is not an enhancer operation. Use enhance, denoise or dereverb.")
        return base
    if MODE == "voice_editor":
        if op not in ("emotion", "voice_characteristics", "whisper", "speed", "custom"):
            raise RuntimeError(f"{op} is not a prompt-editing operation. Check the OPERATION list above.")
        if op == "emotion": base["emotion"] = EMOTION
        elif op == "voice_characteristics": base["description"] = VOICE_DESCRIPTION
        elif op == "speed": base["speed_factor"] = SPEED_FACTOR
        elif op == "custom":
            base["instruction"] = CUSTOM_INSTRUCTION
            base["semantic_contract"] = {"preserve_transcript": True}
        return base
    if op not in ("replace", "insert", "delete"):
        raise RuntimeError(f"{op} is not a transcript operation. Check the OPERATION list above.")
    if not SEGMENT_TEXT.strip() or not EDIT_TARGET_TEXT.strip():
        raise RuntimeError("Transcript edits need SEGMENT_TEXT and EDIT_TARGET_TEXT (checked by listening).")
    if TRANSCRIPT_SCOPE == "whole_clip":
        seg = CHECKED_SOURCE_TEXT
        base.update(segment_start_seconds=0, segment_end_seconds=None, segment_text=seg, segment_scope="whole_clip")
    else:
        seg = SEGMENT_TEXT
        base.update(segment_start_seconds=SEGMENT_START_SECONDS, segment_end_seconds=SEGMENT_END_SECONDS,
                    segment_text=seg, alignment_verified=True)
    if op == "replace":
        if not EDIT_REPLACEMENT_TEXT.strip(): raise RuntimeError("Set EDIT_REPLACEMENT_TEXT.")
        base.update(original_text=EDIT_TARGET_TEXT, replacement_text=EDIT_REPLACEMENT_TEXT,
                    edited_segment_text=seg.replace(EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT, 1))
    elif op == "delete":
        base.update(text=EDIT_TARGET_TEXT,
                    edited_segment_text=" ".join(seg.replace(EDIT_TARGET_TEXT, "", 1).split()))
    else:
        if not INSERT_ANCHOR.strip(): raise RuntimeError("Set INSERT_ANCHOR (a phrase that occurs once in SEGMENT_TEXT).")
        base.update(text=EDIT_TARGET_TEXT, anchor=INSERT_ANCHOR, position=INSERT_POSITION,
                    edited_segment_text=(EDIT_TARGET_TEXT + " " + INSERT_ANCHOR) if INSERT_POSITION == "before" else (INSERT_ANCHOR + " " + EDIT_TARGET_TEXT))
        base["edited_segment_text"] = seg.replace(INSERT_ANCHOR, base["edited_segment_text"], 1)
    return base

if RUN_EDIT:
    if CHECKPOINT is None: raise RuntimeError("Run step 2 first (Drive checkpoint).")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run step 4 first.")
    if not RIGHTS_CONFIRMED: raise RuntimeError("Tick RIGHTS_CONFIRMED only for recordings you have rights to process.")
    if not CHECKED_SOURCE_TEXT.strip(): raise RuntimeError("Set CHECKED_SOURCE_TEXT (your human-checked transcript of the recording).")
    if INPUT_AUDIO.strip():
        SOURCE_FILE = pathlib.Path(INPUT_AUDIO)
        if not SOURCE_FILE.is_file(): raise RuntimeError(f"No file at {SOURCE_FILE}. Leave INPUT_AUDIO empty to upload instead.")
    else:
        from google.colab import files as _files
        print("Upload one recording to edit:")
        _up = _files.upload()
        if not _up: raise RuntimeError("No file uploaded.")
        SOURCE_FILE = pathlib.Path("/content") / next(iter(_up))
    PARAMETERS = build_parameters()
    print("ABOUT TO RUN (check before it uses the GPU):", MODE, "/", OPERATION, "| scope:", TRANSCRIPT_SCOPE if MODE == "transcript_editor" else "n/a")
    print("  source file:", SOURCE_FILE.name)
    for _k in ("source_text", "original_text", "replacement_text", "text", "anchor", "position", "edited_segment_text"):
        if _k in PARAMETERS: print(f"  {_k}: {PARAMETERS[_k]!r}")
    try:
        result = edit_audio(SOURCE_FILE, MODE, OPERATION, PARAMETERS)
    finally:
        print(CHECKPOINT.sync())
    print("RESULT:", result.outcome, "| reason:", result.reason, "| technical_success:", result.technical_success,
          "| automatic_qa_pass:", result.automatic_qa_pass, "| latency_s:", round(result.latency_seconds))
    print("request_dir:", result.request_dir, "(paste this into step 6 to listen again later)")
    for _a in result.attempts:
        _t = (_a.get("qa") or {}).get("checks", {}).get("transcript", {})
        _sp = (_a.get("qa") or {}).get("checks", {}).get("speaker", {})
        print(f"  attempt {_a['attempt']} seed {_a['seed']}: ASR error {_t.get('error_rate')} | speaker {_sp.get('score')} | heard:", repr(_t.get("observed")))
    if result.output_path:
        from IPython.display import Audio, display
        display(Audio(filename=result.output_path))
    print("Human requested-effect/naturalness/seam review remains required.")
else:
    print("Nothing ran. Pick MODE + OPERATION, set CHECKED_SOURCE_TEXT, tick RIGHTS_CONFIRMED and RUN_EDIT, then run this cell.")


In [ ]:
# @title Step 6. Listen again, save a review, or download (no inference)
REQUEST_DIR = "/content/auk-production/shared-requests/b3f53981dbc04d2f8a2916c5fedeebe9"  # @param {type:"string"}
REVIEWER = ""  # @param {type:"string"}
SAVE_REVIEW = True  # @param {type:"boolean"}
QUALITY_PASSED = True  # @param {type:"boolean"}
REVIEW_NOTES = ""  # @param {type:"string"}
if REQUEST_DIR.strip():
    import json
    from IPython.display import Audio, display
    folder=pathlib.Path(REQUEST_DIR);result=json.loads((folder/"result.json").read_text())
    print("Technical:",result["technical_success"],"auto QA:",result["automatic_qa_pass"],"reason:",result["reason"])
    if (folder/"input.wav").is_file():display(Audio(filename=str(folder/"input.wav")))
    if result["output_path"]:display(Audio(filename=result["output_path"]))
    print("Check words, target effect, identity where preserved, speed/timing, naturalness, seams and endings. Review every selected benchmark/acceptance output.")
    if SAVE_REVIEW:
        from auk_reliability.workflow import save_review
        print(save_review(folder,reviewer=REVIEWER,passed=QUALITY_PASSED,notes=REVIEW_NOTES))
else:print("Paste the request_dir printed by step 5 into REQUEST_DIR above.")

if SAVE_REVIEW and CHECKPOINT:print(CHECKPOINT.sync())


## Done? Shut it down

GPU billing only ends with **Runtime > Disconnect and delete runtime** (then check **Manage sessions** shows none). Changing runtime type while disconnected starts a new one.

The editing checks (output validation, bounded retries, transcript and speaker QA) run inside the backend on every request: a failed gate produces no output rather than a bad one.